# OSINT Brasil Flow · teste no navegador 🇧🇷

**Sem instalação no seu computador.** Use **Ambiente de execução → Executar tudo**, autorize a execução do notebook do GitHub e aguarde a instalação. O painel aparece na última célula. Uma sessão **CPU** é suficiente; não é necessário GPU, chave de API, Google Drive ou serviço de túnel.

Este notebook executa o **laboratório Brasil**: grafo, casos, evidências, catálogo e conectores CNPJ → CEP → município. O ambiente completo do Flowsint (Neo4j, PostgreSQL, Redis e editor nativo) continua disponível pela opção Docker no [repositório](https://github.com/Ridd1kulusC0d3r/OsintbrFLOW).

Os arquivos são instalados na máquina temporária do Colab. Ao encerrar ou perder o ambiente, **os casos podem ser apagados**; use a exportação do painel se quiser guardá-los. O acesso usa o proxy do Colab e uma chave aleatória por sessão. Não compartilhe as saídas de uma sessão ativa. O notebook publicado não contém chaves nem resultados.

## 1 · Preparar o ambiente
A primeira execução pode levar vários minutos. Esta etapa baixa o código público, cria um Python isolado, verifica o SHA-256 do Node 24 e compila somente o painel Brasil. Reexecutar atualiza um checkout limpo para a versão atual do repositório e mantém os casos da sessão. Se houver alterações locais no código, a atualização para com uma mensagem, sem sobrescrevê-las.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path
from google.colab import output

if sys.version_info < (3, 12):
    raise RuntimeError("Selecione um ambiente Colab com Python 3.12 ou superior.")

REPOSITORY = "https://github.com/Ridd1kulusC0d3r/OsintbrFLOW.git"
ROOT = Path("/content/OsintbrFLOW")
VENV = Path("/content/osintbrflow-venv")
INSTALL_LOG = Path("/content/osintbrflow-install.log")
DATA = Path("/content/osintbrflow-data")
DATA.mkdir(exist_ok=True)
INSTALL_LOG.write_text("")

def run(command, *, cwd=None, env=None):
    with INSTALL_LOG.open("a") as log:
        result = subprocess.run(command, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        print(INSTALL_LOG.read_text(errors="replace")[-8000:])
        raise RuntimeError(f"Etapa falhou ({result.returncode}). Veja o log acima.")

print("1/4 · Obtendo o repositório…", flush=True)
if not ROOT.exists():
    run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)])
else:
    remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError("A pasta de destino pertence a outro projeto; use um ambiente Colab novo.")
    changes = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
    if changes:
        raise RuntimeError("Há alterações locais no código. Preserve-as antes de atualizar ou use um ambiente novo.")
    run(["git", "fetch", "origin", "main"], cwd=ROOT)
    run(["git", "merge", "--ff-only", "origin/main"], cwd=ROOT)
commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Versão:", commit[:12])

print("2/4 · Preparando Python isolado…", flush=True)
if not (VENV / "bin/python").exists():
    # Colab images may omit ensurepip; virtualenv seeds its own pip.
    bootstrap = Path("/content/osintbrflow-bootstrap")
    run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--target", str(bootstrap), "virtualenv>=20.26,<21"])
    run([sys.executable, "-m", "virtualenv", str(VENV)], env=dict(os.environ, PYTHONPATH=str(bootstrap)))
PYTHON = str(VENV / "bin/python")
run([PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(ROOT / "requirements-brasil.txt")])

print("3/4 · Preparando Node 24…", flush=True)
arch = {"x86_64": "x64", "aarch64": "arm64"}.get(platform.machine())
if not arch:
    raise RuntimeError("Arquitetura não suportada por este notebook.")
node_base = "https://nodejs.org/dist/latest-v24.x/"
with urllib.request.urlopen(node_base + "SHASUMS256.txt", timeout=60) as response:
    checksums = response.read().decode()
expected, filename = next(line.split() for line in checksums.splitlines() if line.endswith(f"-linux-{arch}.tar.xz"))
NODE = Path("/content") / filename.removesuffix(".tar.xz")
if not (NODE / "bin/node").exists():
    archive = Path("/content") / filename
    with urllib.request.urlopen(node_base + filename, timeout=120) as source, archive.open("wb") as target:
        shutil.copyfileobj(source, target)
    with archive.open("rb") as downloaded:
        actual = hashlib.file_digest(downloaded, "sha256").hexdigest()
    if actual != expected:
        raise RuntimeError("Checksum do Node não confere; execute novamente.")
    with tarfile.open(archive) as bundle:
        bundle.extractall("/content", filter="data")
ENV = dict(os.environ, PATH=str(NODE / "bin") + os.pathsep + os.environ["PATH"])
NPM = str(NODE / "bin/npm")

print("4/4 · Instalando e compilando o painel…", flush=True)
run([NPM, "ci", "--ignore-scripts", "--legacy-peer-deps", "--no-audit", "--no-fund"], cwd=ROOT, env=ENV)
run([NPM, "run", "build:brasil", "--workspace", "flowsint-app"], cwd=ROOT, env=ENV)
print("Ambiente pronto. A próxima célula inicia o laboratório.")

## 2 · Iniciar o laboratório
O servidor escuta apenas dentro da máquina do Colab. A chave fica em memória e é enviada pelo painel no cabeçalho de autenticação. A célula pode ser reexecutada: encerra apenas o processo que ela mesma iniciou, preservando os casos enquanto o ambiente existir.

In [ ]:
import secrets
import socket
import time
from urllib.parse import urlsplit

previous = globals().get("osintbr_process")
if previous is not None and previous.poll() is None:
    previous.terminate()
    try:
        previous.wait(timeout=10)
    except subprocess.TimeoutExpired:
        previous.kill()
        previous.wait(timeout=5)

# Select an available port rather than stopping an unrelated notebook service.
with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    PORT = probe.getsockname()[1]
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT}, {{cache: false}})")
proxy = urlsplit(proxy_url)
if proxy.scheme != "https" or not proxy.hostname or proxy.username or proxy.password:
    raise RuntimeError("O Colab não retornou uma URL de proxy HTTPS válida.")
TOKEN = secrets.token_urlsafe(32)
# Reuse this exact origin for the iframe; a second proxyPort call can differ.
PANEL_URL = f"{proxy.scheme}://{proxy.netloc}/brasil.html#token={TOKEN}"
SERVER_LOG = Path("/content/osintbrflow-server.log")
server_env = dict(ENV,
    PYTHONPATH=os.pathsep.join([str(ROOT / "flowsint-types/src"), str(ROOT / "flowsint-core/src")]),
    OSINTBR_FRONTEND=str(ROOT / "flowsint-app/dist-brasil"),
    OSINTBR_DB=str(DATA / "brasil.sqlite3"),
    OSINTBR_PROXY_ORIGIN=f"{proxy.scheme}://{proxy.netloc}",
    OSINTBR_COLAB_TOKEN=TOKEN,
)
with SERVER_LOG.open("w") as log:
    osintbr_process = subprocess.Popen(
        [PYTHON, "-m", "uvicorn", "osintbr.lab:app", "--host", "127.0.0.1", "--port", str(PORT), "--no-access-log"],
        cwd=ROOT, env=server_env, stdout=log, stderr=subprocess.STDOUT,
    )
for attempt in range(60):
    if osintbr_process.poll() is not None:
        raise RuntimeError(SERVER_LOG.read_text(errors="replace")[-4000:])
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as response:
            assert json.load(response)["mode"] == "colab-proxy"
        # Confirm authentication works before displaying the panel.
        request = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/brasil/recipes", headers={"Authorization": "Bearer " + TOKEN})
        with urllib.request.urlopen(request, timeout=2) as response:
            assert response.status == 200
        break
    except (OSError, AssertionError):
        time.sleep(1)
else:
    raise RuntimeError("Servidor não iniciou. Consulte /content/osintbrflow-server.log.")
print("Laboratório ativo. Abra o painel na próxima célula.")

## 3 · Testar
Clique em **Explorar caso demonstrativo** para testar com dados fictícios. Para consultar fontes públicas, crie um caso, escolha **Fontes reais** e informe um CNPJ, CEP ou código de município compatível com o percurso.

Se o painel desaparecer, execute apenas esta célula novamente. Se o ambiente foi desconectado e apagado, use **Executar tudo**. A interface incorporada usa a [API oficial do Colab](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py); o acesso permanece vinculado ao ambiente ativo.

In [ ]:
if osintbr_process.poll() is not None:
    raise RuntimeError("O servidor parou. Reexecute a célula 2.")
output.serve_kernel_port_as_iframe(
    PORT,
    path=PANEL_URL,
    width="100%",
    height=950,
    cache_in_notebook=False,
)

### Encerrar
Para encerrar, use **Ambiente de execução → Desconectar e excluir ambiente de execução**. Exporte os casos que quiser guardar antes disso. Não há download automático para seu computador.